In [1]:
# !pip install spacy
# !python -m spacy download pt_core_news_lg


In [2]:
import pandas as pd
import spacy
from typing import Literal
from spacy.matcher import Matcher
from spacy.tokens import Doc


In [3]:
brasileiras_file_path = 'data/corpus-musicas-brasileiras.csv'
spotify_file_path = 'data/spotify-famosas.csv'

In [4]:
brasileiras_df = pd.read_csv(brasileiras_file_path)
spotify_df = pd.read_csv(spotify_file_path)

In [5]:
#Renomear as colunas para nomes simples
brasileiras_df.columns = ['nome', 'artista', 'genero', 'letra']
brasileiras_df

,nome,artista,genero,letra
0,Carolina,Seu Jorge,MPB,Carolina é uma menina bem difícil de esquecer ...
1,Epitáfio,Titãs,Rock,Devia ter amado mais Ter chorado mais Ter vist...
2,Lugar Ao Sol,Charlie Brown Jr.,Rock,"Que bom viver, como é bom sonhar E o que ficou..."
3,Relicário - Ao Vivo,Cássia Eller,Rock,É uma índia com colar A tarde linda que não qu...
4,Você Me Vira A Cabeça (Me Tira Do Sério),Alcione,Samba,"Você me vira a cabeça, me tira do sério Destró..."
...,...,...,...,...
146607,Hino 96 - a Chave de Ser,Xamã Gideon dos Lakotas,World Music,Esclarecimentos Maiores\nEntrego agora a chave...
146608,Hino 97 - Invocação,Xamã Gideon dos Lakotas,World Music,Invocação Eu Sou\nEu Sou a presença Divina Eu ...
146609,Hino 98 - Conhecimentos,Xamã Gideon dos Lakotas,World Music,Aqui na fazenda ensinamos a você todo conhecim...
146610,Chove Chuva,Miriam Makeba,World Music; Black Music; Blues,Chove chuva\nChove sem parar\nChove chuva\nCho...


In [6]:
spotify_df.columns = ['nome', 'artista', 'ano', 'genero', 'letra', 'playlist']
spotify_df.drop(columns=['ano', 'playlist'], inplace=True)
spotify_df

,nome,artista,genero,letra
0,Carolina,Seu Jorge,MPB,Carolina é uma menina bem difícil de esquecer ...
1,Epitáfio,Titãs,Rock,Devia ter amado mais Ter chorado mais Ter vist...
2,Lugar Ao Sol,Charlie Brown Jr.,Rock,"Que bom viver, como é bom sonhar E o que ficou..."
3,Relicário - Ao Vivo,Cássia Eller,Rock,É uma índia com colar A tarde linda que não qu...
4,Você Me Vira A Cabeça (Me Tira Do Sério),Alcione,Samba,"Você me vira a cabeça, me tira do sério Destró..."
...,...,...,...,...
1622,Carinha de Neném,Japãozin,Piseiro,É o Japãozin da Cachoeira meu patrão É o Japão...
1623,Zero Saudade - Ao Vivo,Os Barões Da Pisadinha,Piseiro,Meu coração desativou você Faz tempo que eu nã...
1624,Meia Noite,Zé Vaqueiro,Piseiro,Você vai deixar a porta aberta para o nosso en...
1625,Recairei - Ao Vivo,Os Barões Da Pisadinha,Piseiro,Já faz uma semana que eu tô limpo de você Iê i...


In [7]:
final_df = pd.concat([brasileiras_df, spotify_df], ignore_index=True)
final_df['music_id'] = final_df.index + 1
final_df

,nome,artista,genero,letra,music_id
0,Carolina,Seu Jorge,MPB,Carolina é uma menina bem difícil de esquecer ...,1
1,Epitáfio,Titãs,Rock,Devia ter amado mais Ter chorado mais Ter vist...,2
2,Lugar Ao Sol,Charlie Brown Jr.,Rock,"Que bom viver, como é bom sonhar E o que ficou...",3
3,Relicário - Ao Vivo,Cássia Eller,Rock,É uma índia com colar A tarde linda que não qu...,4
4,Você Me Vira A Cabeça (Me Tira Do Sério),Alcione,Samba,"Você me vira a cabeça, me tira do sério Destró...",5
...,...,...,...,...,...
148234,Carinha de Neném,Japãozin,Piseiro,É o Japãozin da Cachoeira meu patrão É o Japão...,148235
148235,Zero Saudade - Ao Vivo,Os Barões Da Pisadinha,Piseiro,Meu coração desativou você Faz tempo que eu nã...,148236
148236,Meia Noite,Zé Vaqueiro,Piseiro,Você vai deixar a porta aberta para o nosso en...,148237
148237,Recairei - Ao Vivo,Os Barões Da Pisadinha,Piseiro,Já faz uma semana que eu tô limpo de você Iê i...,148238


In [8]:
final_df.to_csv('data/corpus-musicas.csv', index=False)

In [9]:
import re
#Função para remover caracteres especiais e substituir quebras de linha por ponto
def preprocessar_texto(texto):
    texto = str(texto)  # Garantir que o texto é uma string

    # 1) Substituir quebras de linha únicas por ". "
    texto = re.sub(r"\n(?!\n)", ". ", texto)

    # 2) Injetar ". " entre letra minúscula (incluindo acentos) + espaço + letra maiúscula
    pattern = (
        r'(?<=[a-záàâãéèêíîóôõúûç])'  # atrás: letra minúscula (com acentos)
        r'\s+'                        # um ou mais espaços
        r'(?=[A-ZÁÀÂÃÉÈÊÍÎÓÔÕÚÛÇ])'   # à frente: letra maiúscula (com acentos)
    )
    texto = re.sub(pattern, '. ', texto)

    # 3) Garantir ponto final no fim do texto
    if not texto.endswith('.'):
        texto += '.'

    return texto

final_df['letra_preprocessada'] = final_df['letra'].apply(preprocessar_texto) 
final_df

,nome,artista,genero,letra,music_id,letra_preprocessada
0,Carolina,Seu Jorge,MPB,Carolina é uma menina bem difícil de esquecer ...,1,Carolina é uma menina bem difícil de esquecer....
1,Epitáfio,Titãs,Rock,Devia ter amado mais Ter chorado mais Ter vist...,2,Devia ter amado mais. Ter chorado mais. Ter vi...
2,Lugar Ao Sol,Charlie Brown Jr.,Rock,"Que bom viver, como é bom sonhar E o que ficou...",3,"Que bom viver, como é bom sonhar. E o que fico..."
3,Relicário - Ao Vivo,Cássia Eller,Rock,É uma índia com colar A tarde linda que não qu...,4,É uma índia com colar. A tarde linda que não q...
4,Você Me Vira A Cabeça (Me Tira Do Sério),Alcione,Samba,"Você me vira a cabeça, me tira do sério Destró...",5,"Você me vira a cabeça, me tira do sério. Destr..."
...,...,...,...,...,...,...
148234,Carinha de Neném,Japãozin,Piseiro,É o Japãozin da Cachoeira meu patrão É o Japão...,148235,É o. Japãozin da. Cachoeira meu patrão. É o. J...
148235,Zero Saudade - Ao Vivo,Os Barões Da Pisadinha,Piseiro,Meu coração desativou você Faz tempo que eu nã...,148236,Meu coração desativou você. Faz tempo que eu n...
148236,Meia Noite,Zé Vaqueiro,Piseiro,Você vai deixar a porta aberta para o nosso en...,148237,Você vai deixar a porta aberta para o nosso en...
148237,Recairei - Ao Vivo,Os Barões Da Pisadinha,Piseiro,Já faz uma semana que eu tô limpo de você Iê i...,148238,Já faz uma semana que eu tô limpo de você. Iê ...


In [10]:
final_df.to_csv('data/corpus-musicas-final.csv', index=False)

In [11]:
print(final_df['letra_preprocessada'].iloc[0])


Carolina é uma menina bem difícil de esquecer. Andar bonito e um brilho no olhar. Tem um jeito adolescente que me faz enlouquecer. E um molejo que eu não vou te enganar. Maravilha feminina, meu docinho de pavê. Inteligente, ela é muito sensual. Te confesso que estou apaixonado por você. Ô, Carolina, isso é muito natural. Ô, Carolina, eu preciso de você. Ô, Carolina, eu não vou suportar não te ver. Carolina, eu preciso te falar. Ô, Carolina, eu vou amar você. De segunda a segunda, fico louco pra te ver. Quando eu te ligo, você quase nunca está. Isso era outra coisa que eu queria te dizer. Não temos tempo, então melhor deixar pra lá. A princípio, no domingo, o que você quer fazer? Faça um pedido, que eu irei realizar. Olha aí, amigo, eu digo que ela só me dá prazer. Essa mina. Carolina é de abalar, oi. Ô, Carolina, eu preciso de você. Ô, Carolina, eu não vou suportar não te ver. Carolina, eu preciso te falar. Ô, Carolina eu vou amar você. Carolina, Carolina. Carolina, preciso te encontra

In [12]:
print(final_df.iloc[10000]['letra_preprocessada'])

Sinto que você está dividida. Entre ele e eu. Não sabe o que é melhor pra sua vida. Se é ele ou eu. Vejo que esta dúvida lhe cala. Toda vez que alguém vai lhe perguntar. Quem mais lhe agrada. Nem eu mesmo sei. Eu já fiz de tudo para entender. O que me faz tão parecido assim com ele. O que me falta que você procure nele. O que que eu tenho que ele ainda não lhe deu. Diga qual dos dois tem mais ciúme. O que mais sofre quando você vai embora. Eu só lhe digo que se tem um que lhe adora. Esse alguém sou eu. Se ele tem seus beijos, seus carinhos,. Também tenho eu. Se ele chora quando está sozinho. Também choro eu. Acho que devemos dar um jeito nessa situação. Dois amores dentro do seu peito, e um só coração.


In [13]:
#Quebrando a letra_preprocessada em frases e criando um novo dataframe. Cada música será identificada por seu music_id
def quebrar_em_frases(letra):
    # Usando regex para dividir a letra em frases
    frases = re.split(r'(?<=[.!?]) +', letra)
    return frases

# Iterar sobre cada música e suas letras
all_frases = []
for index, row in final_df.iterrows():
    music_id = row['music_id']
    letra = row['letra_preprocessada']
    # Quebrar a letra em frases
    frases = quebrar_em_frases(letra)
    # Remover frases vazias
    frases = [frase.strip() for frase in frases if frase.strip()]
    for i, frase in enumerate(frases):
        # Adicionar a frase ao DataFrame
        all_frases.append({'music_id': music_id, 'music_frase_id': i + 1, 'frase': frase})
frases_df = pd.DataFrame(all_frases)
frases_df['frase_id'] = frases_df.index + 1

frases_df



,music_id,music_frase_id,frase,frase_id
0,1,1,Carolina é uma menina bem difícil de esquecer.,1
1,1,2,Andar bonito e um brilho no olhar.,2
2,1,3,Tem um jeito adolescente que me faz enlouquecer.,3
3,1,4,E um molejo que eu não vou te enganar.,4
4,1,5,"Maravilha feminina, meu docinho de pavê.",5
...,...,...,...,...
4643921,148239,56,Se eu pego o mundo inteiro.,4643922
4643922,148239,57,Se namorar fosse bom.,4643923
4643923,148239,58,Ninguém tava solteiro.,4643924
4643924,148239,59,Pra quê pegar só uma.,4643925


In [14]:
filtered_frases_df = frases_df.copy()
#Remove frases vazias ou que não possuem letras
filtered_frases_df = filtered_frases_df[filtered_frases_df['frase'].str.strip().astype(bool)]
filtered_frases_df = filtered_frases_df[filtered_frases_df['frase'].str.len() > 0]
filtered_frases_df = filtered_frases_df[filtered_frases_df['frase'].str.contains(r'[a-zA-Z]')]

#Remover frases duplicadas
filtered_frases_df = filtered_frases_df.drop_duplicates(subset=['frase'], keep='first')
filtered_frases_df = filtered_frases_df.reset_index(drop=True)
filtered_frases_df

,music_id,music_frase_id,frase,frase_id
0,1,1,Carolina é uma menina bem difícil de esquecer.,1
1,1,2,Andar bonito e um brilho no olhar.,2
2,1,3,Tem um jeito adolescente que me faz enlouquecer.,3
3,1,4,E um molejo que eu não vou te enganar.,4
4,1,5,"Maravilha feminina, meu docinho de pavê.",5
...,...,...,...,...
2685690,146609,480,Recite as afirmações por 10 minutos ininterrup...,4574304
2685691,146611,4,Chove sem parar\n.,4574434
2685692,146611,10,De molhar o meu divino amor\n.,4574440
2685693,146611,14,Inocente como a flor\n.,4574444


In [15]:
pln = spacy.load('pt_core_news_lg')
pln

In [16]:
def define_pattern(subject_array, gender: Literal["Masc", "Fem"]):
  matcher = Matcher(vocab=pln.vocab)
  gender_morph = f"Gender={gender}|Number=Sing"

  #Sujeitos pré definidos
  sujeitoauxadjetivo = [
    {'LOWER': {'IN': subject_array}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX',  'OP': '?'},  # Verbo auxiliar
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['ADJ']}},  # Adjetivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  sujeitoauxsubst = [
    {'LOWER': {'IN': subject_array}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {'POS': 'AUX'},  # Verbo auxiliar
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': {'IN': ['NOUN']}, "MORPH": {"IN": [gender_morph]}},  # Substantivo
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  sujeitoauxverbo = [
    {'LOWER': {'IN': subject_array}},  # Sujeito feminino
    {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
    {'POS': 'ADV', 'OP': '?'},  # Negação
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX',  'OP': '?'},  # Verbo auxiliar
    {"POS":"VERB", 'OP': '?'},
    {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
    {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
    {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
    {'POS': 'VERB', 'MORPH': {'IN': [f"{gender_morph}|VerbForm=Part|Voice=Pass"]}},  # Verbo na Voz Passiva
    {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
    {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
    {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
    {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  #Nome Próprio
  nomeauxproprio = [
      {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": [gender_morph]}},
      {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
      {'POS': 'ADV', 'OP': '?'},  # Negação
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX'},  # Verbo auxiliar
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
      {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
      {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
      {'POS': {'IN': ['ADJ']}},  # Adjetivo
      {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
      {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
      {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
      {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]


  nomeauxsubst = [
      {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": [gender_morph]}},
      {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
      {'POS': 'ADV', 'OP': '?'},  # Negação
      {'POS': 'AUX'},  # Verbo auxiliar
      {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
      {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
      {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
      {'POS': {'IN': ['NOUN']}, "MORPH": {"IN": [gender_morph]}},  # Substantivo
      {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
      {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
      {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
      {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  nomeauxverbo = [
      {"ENT_TYPE": "PER",'POS': 'PROPN', "MORPH": {"IN": [gender_morph]}},
      {'POS': 'ADV', 'OP': '?'},  # Provavelmente etc
      {'POS': 'ADV', 'OP': '?'},  # Negação
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX'},  # Verbo auxiliar
      {"POS":"VERB", 'OP': '?'},
      {'POS': 'AUX', 'OP': '?'},  # Verbo auxiliar para casos do tipo ela está ficando
      {'POS': 'ADV', 'OP': '?'},  # Realmente / Provavelmente
      {'POS': 'ADV', 'OP': '?'},  # Muito / Pouco
      {'POS': 'VERB', 'MORPH': {'IN': [f"{gender_morph}|VerbForm=Part|Voice=Pass"]}},  # Verbo na Voz Passiva
      {'POS': {'IN': ['ADP', 'SCONJ', 'PART']}, 'OP': '?'},  # Preposição, conjunção ou partícula opcional (ex: "de", "que", "para")
      {'POS': 'DET', 'OP': '?'},  # Determinante opcional (ex: "um", "uma", "o", "a")
      {'POS': {'IN': ['NOUN', 'ADJ']}, 'OP': '?'},  # Substantivo opcional (ex: "marra", "casa", "guerreiro")
      {'POS': 'VERB', 'MORPH': 'VerbForm=Inf', 'OP': '?'},  # Verbo no infinitivo opcional (ex: "viajar", "falar")
  ]

  matcher.add('sujeitoauxadjetivo', patterns=[sujeitoauxadjetivo])
  matcher.add('nomeauxproprio', patterns=[nomeauxproprio])
  matcher.add('sujeitoauxsubst', patterns=[sujeitoauxsubst])
  matcher.add('nomeauxsubst', patterns=[nomeauxsubst])
  matcher.add('sujeitoauxverbo', patterns=[sujeitoauxverbo])
  matcher.add('nomeauxverbo', patterns=[nomeauxverbo])

  return matcher

In [17]:
sujeitos_femininos = ["mulher", "mulheres", "ela", "elas", "menina", "meninas", "garota", "garotas", "senhora", "senhoras", "senhorita", "senhoritas", "moça", "moças", "donzela", "donzelas", "dama", "damas", "rainha", "rainhas", "esposa", "esposas", "namorada", "namoradas", "mina", "minas", "mãe", "mães", "filha", "filhas", "tia", "tias", "avó", "avós", "neta", "netas", "sobrinha", "sobrinhas", "madrasta", "madrastas", "entendeada", "entedeadas", "musa", "musas", "diva", "divas", "deusa", "deusas", "querida", "queridas", "princesa", "princesas"]
sujeitos_masculinos = ["homem", "homens", "ele", "eles", "menino", "meninos", "garoto", "garotos", "senhor", "senhores", "rapaz", "rapazes", "moço", "moços", "cavalheiro", "cavalheiros", "rei", "reis", "marido", "maridos", "namorado", "namorados", "novinho", "novinhos"]

In [18]:
male_matcher = define_pattern(sujeitos_masculinos, "Masc")
female_matcher = define_pattern(sujeitos_femininos, "Fem")

In [19]:
# Percorre frases_df. Pega o conteúdo de frase e aplica o matcher. 
# Caso tenha algum match, poe o valor em uma nova coluna chamada "tem_padrao" e guarda o valor do match em outra coluna chamada "padrao_encontrado"

def apply_matchers(frases_df, male_matcher, female_matcher):
    have_pattern = []
    pattern_found = []
    for index, row in frases_df.iterrows():
        if index % 1000 == 0:
            print(f"Processing index: {index}")
        doc = pln(row['frase'])
        male_matches = male_matcher(doc)
        female_matches = female_matcher(doc)
        if male_matches:
            have_pattern.append("Male")
            pattern_found.append(male_matches)
        elif female_matches:
            have_pattern.append("Female")
            pattern_found.append(female_matches)
        else:
            have_pattern.append("None")
            pattern_found.append(None)
    frases_df['tem_padrao'] = have_pattern
    frases_df['padrao_encontrado'] = pattern_found
    return frases_df
filtered_frases_df = apply_matchers(filtered_frases_df, male_matcher, female_matcher)
filtered_frases_df

Processing index: 0
Processing index: 1000
Processing index: 2000
Processing index: 3000
Processing index: 4000
Processing index: 5000
Processing index: 6000
Processing index: 7000
Processing index: 8000
Processing index: 9000
Processing index: 10000
Processing index: 11000
Processing index: 12000
Processing index: 13000
Processing index: 14000
Processing index: 15000
Processing index: 16000
Processing index: 17000
Processing index: 18000
Processing index: 19000
Processing index: 20000
Processing index: 21000
Processing index: 22000
Processing index: 23000
Processing index: 24000
Processing index: 25000
Processing index: 26000
Processing index: 27000
Processing index: 28000
Processing index: 29000
Processing index: 30000
Processing index: 31000
Processing index: 32000
Processing index: 33000
Processing index: 34000
Processing index: 35000
Processing index: 36000
Processing index: 37000
Processing index: 38000
Processing index: 39000
Processing index: 40000
Processing index: 41000
Proce

,music_id,music_frase_id,frase,frase_id,tem_padrao,padrao_encontrado
0,1,1,Carolina é uma menina bem difícil de esquecer.,1,Female,"[(1916877163388338700, 3, 6), (191687716338833..."
1,1,2,Andar bonito e um brilho no olhar.,2,None,None
2,1,3,Tem um jeito adolescente que me faz enlouquecer.,3,None,None
3,1,4,E um molejo que eu não vou te enganar.,4,None,None
4,1,5,"Maravilha feminina, meu docinho de pavê.",5,None,None
...,...,...,...,...,...,...
2685690,146609,480,Recite as afirmações por 10 minutos ininterrup...,4574304,None,None
2685691,146611,4,Chove sem parar\n.,4574434,None,None
2685692,146611,10,De molhar o meu divino amor\n.,4574440,None,None
2685693,146611,14,Inocente como a flor\n.,4574444,None,None


In [20]:
filtered_frases_df.to_csv('data/frases-musicas-final.csv', index=False)